# LangChain Agent Benchmark 03: Agent Architectures, Memory, and Tools

This notebook uses plain Python, LangChain, and Google Gemini through `langchain-google-genai` to teach agent architectures with real tool calling. The key pattern is: inspect available data, let the model choose and call tools, execute the tool against local data, and ask the model to interpret the actual result.


An **agent** is an LLM running inside a loop of plain text responses (reasoning) and structured output (acting) which continuously append to the context prompt for the next iteration of the loop. 

An LLM capable of **tool usage** is one that can produce structured output (eg. JSON, Pydantic - seen in notebook 01) that can be intercepted in the loop and be exectuted at runtime.

## Native Tool Calling Versus Tool-Like Text

For this notebook we use Google Gemini through `langchain-google-genai`, rather than Huggingface models, because the agent examples need an LLM capable of native tool calling.

Native tool calling means the model does not merely print something like:

```text
Database_Schema("dea_deseq2_results_airway_trt_vs_untrt")
```

Instead, it returns a structured tool-call object. LangChain sees that object, executes the matching Python function, and sends the tool result back to the model. All without human input. 

A plain chat endpoint can still be useful for ordinary text generation, but if it only writes tool-looking text, the tool is not executed. For the examples below, Gemini is used for both plain calls and the real agent so the notebook has a single model provider.


**Reflection Prompts**
- Compare what happens when a model only writes tool-like text versus when it returns an actual structured tool call.
- Identify which parts of the workflow are controlled by the model and which parts are executed deterministically by LangChain.
- Discuss why native tool calling matters when the answer depends on local data rather than model memory.


## Set Up Gemini And Load All Local Data

This notebook uses `langchain-google-genai` because the agent lesson requires native tool calling. With native tool calling, the model returns a structured request to call `Database_Schema`, `Sample_Column_Values`, `SQL_Query`, or `Retrieve_Context`; LangChain then executes the Python tool and sends the result back to the model.

The `GOOGLE_API_KEY` is loaded from the project `.env` file.


**Reflection Prompts**
- Inspect which local data sources are loaded and think about which questions each source can answer.
- Separate model capability from data availability: what can the model know before any tools or files are exposed?


In [1]:
import os
import re
import sqlite3
import time
from pathlib import Path
from IPython.display import Markdown, display

import pandas as pd
import numpy as np
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
import matplotlib.pyplot as plt

load_dotenv()

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

GEMINI_MODEL = "gemini-3.1-flash-lite"

if not os.getenv("GOOGLE_API_KEY"):
    print("Set GOOGLE_API_KEY in the .env file before running Gemini examples.")

model = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL,
    temperature=0,
)

DATA_DIR = Path("../data")
if not DATA_DIR.exists():
    DATA_DIR = Path("data")

DOCS_DIR = Path("../docs")
if not DOCS_DIR.exists():
    DOCS_DIR = Path("docs")

DB_PATH = Path("../agent_teaching.sqlite")
if not Path("../data").exists():
    DB_PATH = Path("agent_teaching.sqlite")
DB_PATH = DB_PATH.resolve()
DB_READONLY_URI = DB_PATH.as_uri() + "?mode=ro"
if DB_PATH.exists():
    DB_PATH.unlink()

all_data_files = sorted(
    path
    for path in DATA_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in [".csv", ".txt"]
)

print("Files used as local information:")
for path in all_data_files:
    print("-", path)

print("\nSQLite database file:", DB_PATH)


Files used as local information:
- ../data/airway_counts.csv
- ../data/airway_metadata.csv
- ../data/dea/DESeq2_results_airway_trt_vs_untrt.csv
- ../data/functional/gProfiler_ORA_results.txt
- ../data/functional/topGO_results_BP_all_DE.txt
- ../data/functional/topGO_results_BP_downregulated.txt
- ../data/functional/topGO_results_BP_upregulated.txt

SQLite database file: /Users/camilla.callierotti/Library/CloudStorage/OneDrive-Htechnopole/Conferences/2026_piacenza/agent_teaching.sqlite


In [2]:
tables = {}
text_documents = []
manifest_rows = []

for path in all_data_files:
    relative_path = path.relative_to(DATA_DIR)
    # SQLite table names come from file paths, so normalize them once and use the same names everywhere downstream.
    table_name = "__".join(relative_path.with_suffix("").parts)
    table_name = re.sub(r"[^0-9a-zA-Z]+", "_", table_name).strip("_").lower()

    file_info = {
        "file": str(path),
        "table_name": table_name,
        "rows": None,
        "columns": None,
        "loaded_as": None,
        "notes": "",
    }

    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
        # Normalize headers so SQL queries and prompts can use predictable lowercase column names.
        df.columns = [
            re.sub(r"[^0-9a-zA-Z]+", "_", str(column)).strip("_").lower() or "unnamed_0"
            for column in df.columns
        ]
        if "unnamed_0" in df.columns:
            if path.name == "airway_metadata.csv":
                # This CSV stores sample IDs in its unnamed index column; name it so SQL questions can refer to it clearly.
                df = df.rename(columns={"unnamed_0": "sample_id"})
            else:
                df = df.drop(columns=["unnamed_0"])

        tables[table_name] = df
        file_info["rows"] = len(df)
        file_info["columns"] = len(df.columns)
        file_info["loaded_as"] = "table"
        file_info["notes"] = "CSV loaded with pandas."
        text_documents.append({
            "source": str(path),
            "text": "Source: " + str(path) + "\nTable: " + table_name + "\nColumns: " + ", ".join(df.columns) + "\nPreview:\n" + df.head(5).to_string(index=False),
        })

    elif path.suffix.lower() == ".txt":
        df = pd.read_csv(path, sep="\t")
        # Normalize headers so SQL queries and prompts can use predictable lowercase column names.
        df.columns = [
            re.sub(r"[^0-9a-zA-Z]+", "_", str(column)).strip("_").lower() or "unnamed_0"
            for column in df.columns
        ]
        tables[table_name] = df
        file_info["rows"] = len(df)
        file_info["columns"] = len(df.columns)
        file_info["loaded_as"] = "table"
        file_info["notes"] = "TXT loaded with pandas as a tab-separated table."
        text_documents.append({
            "source": str(path),
            "text": "Source: " + str(path) + "\nTable: " + table_name + "\nColumns: " + ", ".join(df.columns) + "\nPreview:\n" + df.head(5).to_string(index=False),
        })

    manifest_rows.append(file_info)

manifest = pd.DataFrame(manifest_rows)
df = manifest
df.style.set_properties(
    subset=["notes"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


,file,table_name,rows,columns,loaded_as,notes
0,../data/airway_counts.csv,airway_counts,63677,18,table,CSV loaded with pandas.
1,../data/airway_metadata.csv,airway_metadata,8,10,table,CSV loaded with pandas.
2,../data/dea/DESeq2_results_airway_trt_vs_untrt.csv,dea_deseq2_results_airway_trt_vs_untrt,22369,10,table,CSV loaded with pandas.
3,../data/functional/gProfiler_ORA_results.txt,functional_gprofiler_ora_results,745,13,table,TXT loaded with pandas as a tab-separated table.
4,../data/functional/topGO_results_BP_all_DE.txt,functional_topgo_results_bp_all_de,7450,6,table,TXT loaded with pandas as a tab-separated table.
5,../data/functional/topGO_results_BP_downregulated.txt,functional_topgo_results_bp_downregulated,7450,6,table,TXT loaded with pandas as a tab-separated table.
6,../data/functional/topGO_results_BP_upregulated.txt,functional_topgo_results_bp_upregulated,7450,6,table,TXT loaded with pandas as a tab-separated table.


In [3]:
connection = sqlite3.connect(str(DB_PATH))

for table_name in tables:
    tables[table_name].to_sql(table_name, connection, index=False, if_exists="replace")

connection.commit()

print("Tables available for SQL:")
for table_name in sorted(tables):
    print("-", table_name, tables[table_name].shape)

# Build a schema description for the LLM.
# This is the equivalent of a Database_Schema tool.
table_names = pd.read_sql_query(
    "select name from sqlite_master where type = 'table' order by name",
    connection,
)

schema_text = ""
for table_name in table_names["name"]:
    columns = pd.read_sql_query("pragma table_info(" + table_name + ")", connection)
    schema_text = schema_text + "Table: " + table_name + "\n"
    for _, column in columns.iterrows():
        schema_text = schema_text + "- " + column["name"] + " (" + column["type"] + ")\n"
    schema_text = schema_text + "\n"

print("Schema text is ready for natural-language-to-SQL prompts.")

connection.close()
del connection


Tables available for SQL:
- airway_counts (63677, 18)
- airway_metadata (8, 10)
- dea_deseq2_results_airway_trt_vs_untrt (22369, 10)
- functional_gprofiler_ora_results (745, 13)
- functional_topgo_results_bp_all_de (7450, 6)
- functional_topgo_results_bp_downregulated (7450, 6)
- functional_topgo_results_bp_upregulated (7450, 6)
Schema text is ready for natural-language-to-SQL prompts.


In [4]:
# Add the paper PDF to the same simple text collection.
try:
    from pypdf import PdfReader
    pdf_path = DOCS_DIR / "paper.pdf"
    reader = PdfReader(str(pdf_path))
    for page_number, page in enumerate(reader.pages):
        page_text = page.extract_text()
        if page_text:
            text_documents.append({
                "source": str(pdf_path) + " page " + str(page_number + 1),
                "text": page_text[:4000],
            })
    print("Loaded paper pages:", len(reader.pages))
except Exception as exc:
    print("The paper PDF was not loaded:", repr(exc))

print("Text chunks available for retrieval:", len(text_documents))


Loaded paper pages: 13
Text chunks available for retrieval: 20


## 1. Plain LLM

A plain LLM call gets only the question. It does not see the local files, so it may guess.


**Reflection Prompts**
- Compare the plain LLM answer with later tool-based answers: where does it guess, hedge, or stay generic?
- Identify claims that would require access to the local files before they could be trusted.
- Decide whether fluency makes the answer seem more reliable than the evidence actually supports.


In [42]:
question = "Which local files would an agent need to answer: sample composition, top dexamethasone-induced genes, and enriched biological processes?"
chunks = []
usage = {}
start = time.perf_counter()
for chunk in model.stream(question):
    if chunk.content:
        text = chunk.text()
        chunks.append(text)
        print(text, end="")
    if chunk.usage_metadata:
        usage = chunk.usage_metadata

answer = "".join(chunks)
output_token_details = usage.get("output_token_details", {}) if usage else {}
print("\n\nseconds:", round(time.perf_counter() - start, 3))
print("prompt_tokens:", usage.get("input_tokens"))
print("completion_tokens:", usage.get("output_tokens"))
print("total_tokens:", usage.get("total_tokens"))
print("reasoning_tokens:", output_token_details.get("reasoning") or output_token_details.get("reasoning_tokens"))
print("cost:", usage.get("cost"))


/opt/miniconda3/envs/agent_teaching_py312/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  exec(code_obj, self.user_global_ns, self.user_ns)


To answer these questions, an agent would typically need access to a combination of **raw data files** (for processing) and **metadata/annotation files** (for interpretation).

Depending on the specific bioinformatics pipeline used, here are the files required:

### 1. For "Sample Composition"
This refers to the metadata describing the experimental design (e.g., which samples are treated vs. control, cell types, time points).
*   **Sample Metadata File (`metadata.csv` or `samplesheet.tsv`):** A table mapping sample IDs to their experimental conditions.
*   **Raw Data Files (FASTQ or BAM):** If the agent needs to verify the composition (e.g., checking read counts or mapping quality), it would need access to the raw sequencing files or the alignment files.
*   **Count Matrix (`counts.csv`):** A summary file showing the number of reads per gene per sample, which confirms the final set of samples included in the analysis.

### 2. For "Top Dexamethasone-Induced Genes"
This requires the resu

The LLMs internal knowledge knows what it needs to answer the question, but it doesnt have the *tools* to execute it. Therefore, we can harness its knowledge to let it reason on its own, and only provide it the data and tools for runtime exectution.

## Create Real Agent Tools

This is the key agent pattern used in the omics-query solution.

The user asks one natural-language question. The agent must decide which tools to call:

1. `Database_Schema` to inspect tables and columns.
2. `Sample_Column_Values` to inspect real categorical/text values before filtering.
3. `SQL_Query` to execute a generated read-only SQL query.
4. `Retrieve_Context` when it needs paper or file context.

The Python code below does not hard-code `padj < 0.05`, `padj < 0.01`, or `CRISPLD2` answers. Those are inside the user's natural-language question; the agent has to turn them into tool calls.

In a notebook, the agent tools open a fresh read-only SQLite connection inside each tool call. That avoids the SQLite thread error that happens when a connection created in one notebook thread is reused by a tool running in another thread.


**Reflection Prompts**
- Map each tool to the type of evidence it can provide: schema, values, SQL results, or text context.
- Ask which tool calls should happen before answering a question about specific genes, thresholds, or metadata.
- Consider how tool boundaries reduce risk compared with letting the model invent file contents or SQL results.


In [5]:
@tool
def Database_Schema(input_text: str = "") -> str:
    """Return all available SQLite tables and their columns. Use this before writing SQL."""
    with sqlite3.connect(DB_READONLY_URI, uri=True) as tool_connection:
        table_names = pd.read_sql_query(
            "select name from sqlite_master where type = 'table' order by name",
            tool_connection,
        )

    output = "Available tables and schemas:\n\n"
    for table_name in table_names["name"]:
        with sqlite3.connect(DB_READONLY_URI, uri=True) as tool_connection:
            columns = pd.read_sql_query("pragma table_info(" + table_name + ")", tool_connection)
        output = output + "Table: " + table_name + "\n"
        for _, column in columns.iterrows():
            output = output + "- " + column["name"] + " (" + column["type"] + ")\n"
        output = output + "\n"
    return output


@tool
def Sample_Column_Values(input_text: str = "") -> str:
    """Return example distinct values from text columns, so the agent does not guess categorical values."""
    with sqlite3.connect(DB_READONLY_URI, uri=True) as tool_connection:
        table_names = pd.read_sql_query(
            "select name from sqlite_master where type = 'table' order by name",
            tool_connection,
        )

    output = "Example values from text columns:\n\n"
    for table_name in table_names["name"]:
        with sqlite3.connect(DB_READONLY_URI, uri=True) as tool_connection:
            columns = pd.read_sql_query("pragma table_info(" + table_name + ")", tool_connection)
        for _, column in columns.iterrows():
            column_name = column["name"]
            column_type = str(column["type"]).lower()
            if "text" in column_type:
                try:
                    query = 'select distinct "' + column_name + '" from "' + table_name + '" where "' + column_name + '" is not null limit 8'
                    with sqlite3.connect(DB_READONLY_URI, uri=True) as tool_connection:
                        values = pd.read_sql_query(query, tool_connection)
                    if len(values) > 0:
                        output = output + table_name + "." + column_name + ": " + ", ".join(values[column_name].astype(str).tolist()) + "\n"
                except Exception:
                    pass
    return output


@tool
def SQL_Query(query: str) -> str:
    """Execute one read-only SQLite SELECT query and return the actual rows. Input must be a complete SQL SELECT statement."""
    query_clean = query.strip().rstrip(";")
    if not query_clean.lower().startswith("select"):
        return "Rejected: only SELECT queries are allowed."
    forbidden_words = ["drop", "delete", "insert", "update", "alter", "create", "replace"]
    for word in forbidden_words:
        if word in query_clean.lower().split():
            return "Rejected: this query contains a forbidden SQL keyword."
    try:
        with sqlite3.connect(DB_READONLY_URI, uri=True) as tool_connection:
            result = pd.read_sql_query(query_clean, tool_connection)
        if len(result) == 0:
            return "Query returned 0 rows."
        return "Query returned " + str(len(result)) + " rows. Showing up to 30 rows:\n" + result.head(30).to_string(index=False)
    except Exception as exc:
        return "SQL error: " + repr(exc)


@tool
def Retrieve_Context(query: str) -> str:
    """Retrieve relevant local file or paper context using simple keyword matching."""
    query_words = query.lower().split()
    scored_documents = []
    for document in text_documents:
        text = document["text"].lower()
        score = 0
        for word in query_words:
            score = score + text.count(word)
        scored_documents.append({"score": score, "source": document["source"], "text": document["text"]})
    scored_documents = sorted(scored_documents, key=lambda row: row["score"], reverse=True)

    output = ""
    for document in scored_documents[:5]:
        output = output + "\n\nSOURCE: " + document["source"] + "\n" + document["text"][:1500]
    return output

@tool
def Volcano_Plot(padj_cutoff: float = 0.05, lfc_cutoff: float = 1.0, top_n_labels: int = 10) -> str:
    """Create a volcano plot from the DESeq2 results table and return the saved image path."""
    query = """
    select gene_name, symbol, log2foldchange, padj
    from dea_deseq2_results_airway_trt_vs_untrt
    where log2foldchange is not null
      and padj is not null
    """

    with sqlite3.connect(DB_READONLY_URI, uri=True) as tool_connection:
        df = pd.read_sql_query(query, tool_connection)

    df["minus_log10_padj"] = -np.log10(df["padj"].clip(lower=np.nextafter(0, 1)))
    df["significant"] = (df["padj"] < padj_cutoff) & (df["log2foldchange"].abs() >= lfc_cutoff)

    output_dir = Path("agent_outputs")
    output_dir.mkdir(exist_ok=True)
    output_path = output_dir / "volcano_plot.png"

    plt.figure(figsize=(7, 5))
    plt.scatter(
        df["log2foldchange"],
        df["minus_log10_padj"],
        c=df["significant"].map({True: "crimson", False: "lightgray"}),
        s=12,
        alpha=0.75,
        linewidths=0,
    )

    plt.axvline(-lfc_cutoff, color="black", linestyle="--", linewidth=1)
    plt.axvline(lfc_cutoff, color="black", linestyle="--", linewidth=1)
    plt.axhline(-np.log10(padj_cutoff), color="black", linestyle="--", linewidth=1)

    label_df = df.sort_values("padj").head(top_n_labels)
    for _, row in label_df.iterrows():
        label = row["symbol"] or row["gene_name"]
        plt.text(row["log2foldchange"], row["minus_log10_padj"], label, fontsize=8)

    plt.xlabel("log2 fold change")
    plt.ylabel("-log10 adjusted p-value")
    plt.title("Volcano plot: dexamethasone vs untreated")
    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()

    return f"Saved volcano plot to {output_path}"

agent_tools = [
    Database_Schema,
    Sample_Column_Values,
    SQL_Query,
    Retrieve_Context,
    Volcano_Plot,
]

print("Tools available to the agent:")
for agent_tool in agent_tools:
    print("-", agent_tool.name)


Tools available to the agent:
- Database_Schema
- Sample_Column_Values
- SQL_Query
- Retrieve_Context
- Volcano_Plot


## Build The Agent

This is the actual agent. The notebook does not write the SQL query for the question. The agent receives the natural-language question and has to use its tools.

The agent is composed of:
- Model: the LLM which reasons and orchestrates
- Agent tools: the tools the LLM has at its disposal which we have coded above
- System prompt: general guidance on how to act from how to use the data, how to use the tools, and how to format its final response

**Reflection Prompts**
- Identify the responsibilities of the model, the tools, and the system prompt in the final agent behavior.
- Compare this architecture with a plain function call: what flexibility is gained, and what uncertainty is introduced?
- Look for where the system prompt constrains the agent and where the agent still has freedom to choose its path.


In [6]:
system_prompt = """
You are an RNA-seq teaching agent.

Answer questions by using tools, not by guessing.

Required workflow for data questions:
1. First call Database_Schema to inspect available tables and columns.
2. Then call Sample_Column_Values before filtering text/categorical columns such as gene symbols, treatment labels, GO terms, or file-derived labels.
3. Then write the correct SQLite SELECT query and call SQL_Query.
4. If biological interpretation or source context is needed, call Retrieve_Context.
5. Give a final answer based only on tool results from this turn.

Important table guidance for this notebook:
- dea_deseq2_results_airway_trt_vs_untrt contains differential expression columns including gene_id, symbol, log2foldchange, pvalue, and padj.
- airway_metadata contains sample information including sample_id, cell, and dex.
- airway_counts contains gene annotation columns plus sample-count columns named like srr1039508.
- functional_topgo_results_bp_upregulated, functional_topgo_results_bp_downregulated, functional_topgo_results_bp_all_de, and functional_gprofiler_ora_results contain enrichment results.
- Use padj for adjusted p-value thresholds in dea_deseq2_results_airway_trt_vs_untrt.
- Use SELECT only. Never modify data.
- Always include a short biological interpretation and a caveat.
"""

rna_seq_agent = create_agent(
    model,
    agent_tools,
    system_prompt=system_prompt,
)

print("Agent is ready. This version uses Gemini native tool calling through langchain-google-genai.")


Agent is ready. This version uses Gemini native tool calling through langchain-google-genai.


## Ask Natural-Language Questions

Now the user can ask normal questions. The agent should inspect schema, write SQL, run SQL, and answer.

**Note on tool design**

Tools can be written so that some plotting choices are fixed by the developer, while other choices are exposed to the agent and can be changed through natural language.

For example, in the `Volcano_Plot` tool, we may want to keep some visual conventions fixed: the x-axis is always `log2foldchange`, the y-axis is always `-log10(padj)`, significant genes are always highlighted in the same color, and the output is always saved to the same folder. These fixed choices make the plot consistent and reduce ambiguity.

Other choices can be made configurable by adding tool arguments. For example, `padj_cutoff`, `lfc_cutoff`, and `top_n_labels` can be parameters of the tool. Then a user can ask: “Plot a volcano plot with padj < 0.01 and label the top 20 genes,” and the agent can translate that request into a structured tool call with those argument values.

This separation is important: the tool controls the reliable, reproducible parts of the analysis, while natural language controls the flexible parts of the request. In practice, good tool design means deciding which parameters users should be allowed to change, and which defaults should remain fixed for clarity, consistency, or scientific correctness.


**Reflection Prompts**
- Trace the agent path from user question to tool calls to final answer.
- Check whether the final answer is supported by the tool observations shown in the notebook output.
- Notice whether the agent asks the right intermediate questions before producing a conclusion.


In [8]:
questions = [
    "How many genes are significant at padj < 0.05 versus padj < 0.01?",
    "Plot a volcano plot of the significant genes at padj < 0.05 and label the top 10 genes by padj.",
]

for question in questions:
    chunks = []
    usage = {}
    start = time.perf_counter()
    markdown_text = chr(10).join(["### Question", question, "", "_Streaming agent answer..._"])
    answer_display = display(Markdown(markdown_text), display_id=True)

    for event in rna_seq_agent.stream(
        {"messages": [{"role": "user", "content": question}]},
        stream_mode="values",
    ):
        final_message = event["messages"][-1]
        tool_calls = getattr(final_message, "tool_calls", None)
        if tool_calls:
            print("tool_calls:", tool_calls)

        content = final_message.content
        answer_text = ""
        if isinstance(content, list):
            for block in content:
                if isinstance(block, dict) and block.get("type") == "text":
                    answer_text = answer_text + block.get("text", "") + chr(10) + chr(10)
                else:
                    answer_text = answer_text + str(block) + chr(10) + chr(10)
        else:
            answer_text = str(content)

        chunks = [answer_text.strip()]
        markdown_text = chr(10).join(["### Question", question, "", "### Agent answer", "".join(chunks)])
        answer_display.update(Markdown(markdown_text))

        if getattr(final_message, "usage_metadata", None):
            usage = final_message.usage_metadata

    output_token_details = usage.get("output_token_details", {}) if usage else {}
    print("seconds:", round(time.perf_counter() - start, 3))
    print("prompt_tokens:", usage.get("input_tokens"))
    print("completion_tokens:", usage.get("output_tokens"))
    print("total_tokens:", usage.get("total_tokens"))
    print("reasoning_tokens:", output_token_details.get("reasoning") or output_token_details.get("reasoning_tokens"))
    print("cost:", usage.get("cost"))
    print()

### Question
How many genes are significant at padj < 0.05 versus padj < 0.01?

### Agent answer
There are **4,000** genes significant at an adjusted p-value (padj) < 0.05, and **2,901** genes significant at a more stringent padj < 0.01.

**Biological Interpretation:**
The higher number of significant genes at the 0.05 threshold indicates that relaxing the significance criteria captures a broader set of potentially differentially expressed genes, likely including those with smaller effect sizes or higher variability. The 2,901 genes at the 0.01 threshold represent a more confident set of genes with stronger evidence of differential expression between the treated and untreated conditions.

**Caveat:**
These counts are based on the Benjamini-Hochberg adjusted p-values (padj), which control the False Discovery Rate (FDR). While these thresholds are standard, the choice of threshold should ideally be balanced against the specific goals of the downstream analysis (e.g., prioritizing high-confidence candidates vs. capturing a comprehensive pathway signature).

tool_calls: [{'name': 'Database_Schema', 'args': {}, 'id': 'eNcHrQqe', 'type': 'tool_call'}]
tool_calls: [{'name': 'SQL_Query', 'args': {'query': 'SELECT \n    SUM(CASE WHEN padj < 0.05 THEN 1 ELSE 0 END) AS count_0_05,\n    SUM(CASE WHEN padj < 0.01 THEN 1 ELSE 0 END) AS count_0_01\nFROM dea_deseq2_results_airway_trt_vs_untrt;'}, 'id': 'KT455DtC', 'type': 'tool_call'}]
seconds: 2.774
prompt_tokens: 1454
completion_tokens: 210
total_tokens: 1664
reasoning_tokens: None
cost: None



### Question
Plot a volcano plot of the significant genes at padj < 0.05 and label the top 10 genes by padj.

### Agent answer
The volcano plot has been generated and saved to `agent_outputs/volcano_plot.png`. This plot displays the log2 fold change on the x-axis and the -log10 adjusted p-value on the y-axis, with genes meeting the significance threshold of `padj < 0.05` highlighted. The top 10 most significant genes (based on the lowest adjusted p-values) are labeled.

**Biological Interpretation:**
The volcano plot allows for the rapid identification of genes that are both statistically significant and biologically relevant (high fold change). Genes in the upper-right quadrant are significantly upregulated, while those in the upper-left are significantly downregulated.

**Caveat:**
The labeling of the "top 10" genes is based strictly on the adjusted p-value. While these genes are the most statistically significant, they may not necessarily be the genes with the largest biological effect size (log2 fold change). Always consider both magnitude and significance when interpreting differential expression results.

tool_calls: [{'name': 'Volcano_Plot', 'args': {'padj_cutoff': 0.05, 'top_n_labels': 10}, 'id': 'hExOSyzF', 'type': 'tool_call'}]
seconds: 3.159
prompt_tokens: 698
completion_tokens: 208
total_tokens: 906
reasoning_tokens: None
cost: None



## More Natural-Language Questions

These examples use the same tools. The notebook does not change Python code to handle each question.


**Reflection Prompts**
- Compare how the same agent handles different question types without changing the Python code.
- Identify which questions require SQL, retrieval, or both.
- Evaluate whether the agent adapts its strategy to the question or follows a repeated pattern mechanically.


In [46]:
questions = [
    "What is the DESeq2 result for CRISPLD2?",
    "Which 10 genes have the largest positive log2FoldChange among genes with padj < 0.01?",
    "Which biological processes are enriched among upregulated genes?",
]

for question in questions:
    chunks = []
    usage = {}
    start = time.perf_counter()
    markdown_text = chr(10).join(["### Question", question, "", "_Streaming agent answer..._"])
    answer_display = display(Markdown(markdown_text), display_id=True)

    for event in rna_seq_agent.stream({"messages": [{"role": "user", "content": question}]}, stream_mode="values"):
        final_message = event["messages"][-1]
        tool_calls = getattr(final_message, "tool_calls", None)
        if tool_calls:
            print("tool_calls:", tool_calls)
        content = final_message.content
        answer_text = ""
        if isinstance(content, list):
            for block in content:
                if isinstance(block, dict) and block.get("type") == "text":
                    answer_text = answer_text + block.get("text", "") + chr(10) + chr(10)
        else:
            answer_text = str(content)
        chunks = [answer_text.strip()]
        markdown_text = chr(10).join(["### Question", question, "", "### Agent answer", "".join(chunks)])
        answer_display.update(Markdown(markdown_text))
        if getattr(final_message, "usage_metadata", None):
            usage = final_message.usage_metadata

    output_token_details = usage.get("output_token_details", {}) if usage else {}
    print("seconds:", round(time.perf_counter() - start, 3))
    print("prompt_tokens:", usage.get("input_tokens"))
    print("completion_tokens:", usage.get("output_tokens"))
    print("total_tokens:", usage.get("total_tokens"))
    print("reasoning_tokens:", output_token_details.get("reasoning") or output_token_details.get("reasoning_tokens"))
    print("cost:", usage.get("cost"))
    display(Markdown("---"))


### Question
What is the DESeq2 result for CRISPLD2?

### Agent answer
The DESeq2 results for the gene **CRISPLD2** in the airway treatment vs. untreated comparison are as follows:

*   **log2FoldChange:** 2.63
*   **p-value:** 2.45e-48
*   **padj (Adjusted p-value):** 7.88e-46
*   **baseMean:** 3082.09

**Biological Interpretation:**
CRISPLD2 is significantly upregulated in the treated airway smooth muscle cells, with a large positive log2 fold change and a highly significant adjusted p-value. This gene is known to be glucocorticoid-responsive and is often associated with anti-inflammatory pathways in airway tissues.

**Caveat:**
While the statistical significance is very high, these results are based on the specific experimental conditions (dexamethasone treatment of airway smooth muscle cells) provided in the dataset. Results may vary in different cell types or under different experimental conditions.

tool_calls: [{'name': 'Database_Schema', 'args': {}, 'id': 'zHFYXdEn', 'type': 'tool_call'}]
tool_calls: [{'name': 'SQL_Query', 'args': {'query': "SELECT * FROM dea_deseq2_results_airway_trt_vs_untrt WHERE symbol = 'CRISPLD2'"}, 'id': 'Y7jF6wp8', 'type': 'tool_call'}]
seconds: 4.377
prompt_tokens: 1495
completion_tokens: 211
total_tokens: 1706
reasoning_tokens: None
cost: None


---

### Question
Which 10 genes have the largest positive log2FoldChange among genes with padj < 0.01?

### Agent answer
The 10 genes with the largest positive log2FoldChange (indicating the strongest upregulation in the treated group compared to the untreated group) that meet the significance threshold of padj < 0.01 are:

| Gene Symbol | log2FoldChange | padj |
| :--- | :--- | :--- |
| ALOX15B | 9.51 | 1.18e-17 |
| ZBTB16 | 7.35 | 2.32e-40 |
| RP11-357D18.1 | 6.33 | 6.03e-19 |
| GUCY2D | 5.88 | 8.86e-05 |
| STEAP4 | 5.21 | 5.06e-24 |
| RP11-434D9.1 | 5.09 | 1.13e-04 |
| ANGPTL7 | 5.08 | 5.18e-10 |
| PRODH | 4.89 | 4.09e-19 |
| LINC00890 | 4.88 | 4.11e-04 |
| FAM107A | 4.83 | 6.53e-46 |

**Biological Interpretation:** These genes show the most dramatic increase in expression following the treatment. Genes like *ALOX15B* and *ZBTB16* are known to be highly responsive to glucocorticoid treatment in airway smooth muscle cells.

**Caveat:** While these genes have the highest fold changes, high fold change does not always equate to high biological relevance; it is important to consider the base expression levels (baseMean) and the specific biological context of the experiment.

tool_calls: [{'name': 'Database_Schema', 'args': {'input_text': 'dea_deseq2_results_airway_trt_vs_untrt'}, 'id': 'C7RvRH0w', 'type': 'tool_call'}]
tool_calls: [{'name': 'SQL_Query', 'args': {'query': 'SELECT symbol, log2foldchange, padj FROM dea_deseq2_results_airway_trt_vs_untrt WHERE padj < 0.01 ORDER BY log2foldchange DESC LIMIT 10;'}, 'id': 'G5FXPpeQ', 'type': 'tool_call'}]
seconds: 4.318
prompt_tokens: 1731
completion_tokens: 416
total_tokens: 2147
reasoning_tokens: None
cost: None


---

### Question
Which biological processes are enriched among upregulated genes?

### Agent answer
The biological processes enriched among the upregulated genes in the airway dataset (dexamethasone-treated vs. untreated) are primarily related to cell signaling, adhesion, and structural remodeling. The top enriched terms include:

*   **Cell adhesion** (GO:0007155)
*   **Angiogenesis** (GO:0001525)
*   **Positive regulation of cell migration** (GO:0030335)
*   **Cellular response to hypoxia** (GO:0071456)

**Biological Interpretation:**
The enrichment of these processes suggests that dexamethasone treatment in airway smooth muscle cells significantly alters the extracellular environment and cell-cell interactions, potentially impacting tissue remodeling and vascularization responses.

**Caveat:**
These results are based on Gene Ontology (GO) enrichment analysis using the `weight01` algorithm, which accounts for the hierarchical structure of GO terms to reduce redundancy. However, enrichment analysis only identifies overrepresented categories and does not confirm the functional directionality or the specific physiological outcome of these changes in a clinical context.

tool_calls: [{'name': 'Database_Schema', 'args': {}, 'id': 'V3hyqprR', 'type': 'tool_call'}]
tool_calls: [{'name': 'SQL_Query', 'args': {'query': 'SELECT * FROM functional_topgo_results_bp_upregulated ORDER BY weight01 ASC LIMIT 10;'}, 'id': 'qrRQ8Xkl', 'type': 'tool_call'}]
seconds: 2.246
prompt_tokens: 1739
completion_tokens: 232
total_tokens: 1971
reasoning_tokens: None
cost: None


---

## Architecture Comparison

The remaining cells compare architectural ideas using the same real tools. The important point is that the agent receives natural language and decides how to use `Database_Schema`, `Sample_Column_Values`, `SQL_Query`, and `Retrieve_Context`.


**Reflection Prompts**
- Compare architectures by reliability, transparency, flexibility, latency, and ease of debugging.
- Ask which architecture you would choose for exploratory analysis versus a production pipeline.
- Look for cases where a simpler architecture may be safer than a more autonomous agent.


In [47]:
question = "Compare significant genes at padj < 0.05 and padj < 0.01, then explain what this means biologically."
chunks = []
usage = {}
start = time.perf_counter()
markdown_text = chr(10).join(["### Tool-using agent", "", "**Question**", question, "", "_Streaming agent answer..._"])
answer_display = display(Markdown(markdown_text), display_id=True)

for event in rna_seq_agent.stream({"messages": [{"role": "user", "content": question}]}, stream_mode="values"):
    final_message = event["messages"][-1]
    tool_calls = getattr(final_message, "tool_calls", None)
    if tool_calls:
        print("tool_calls:", tool_calls)
    content = final_message.content
    answer_text = ""
    if isinstance(content, list):
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                answer_text = answer_text + block.get("text", "") + "\n\n"
            else:
                answer_text = answer_text + str(block) + "\n\n"
    else:
        answer_text = str(content)
    chunks = [answer_text.strip()]
    markdown_text = chr(10).join(["### Tool-using agent", "", "**Question**", question, "", "".join(chunks)])
    answer_display.update(Markdown(markdown_text))
    if getattr(final_message, "usage_metadata", None):
        usage = final_message.usage_metadata

output_token_details = usage.get("output_token_details", {}) if usage else {}
print("seconds:", round(time.perf_counter() - start, 3))
print("prompt_tokens:", usage.get("input_tokens"))
print("completion_tokens:", usage.get("output_tokens"))
print("total_tokens:", usage.get("total_tokens"))
print("reasoning_tokens:", output_token_details.get("reasoning") or output_token_details.get("reasoning_tokens"))
print("cost:", usage.get("cost"))


### Tool-using agent

**Question**
Compare significant genes at padj < 0.05 and padj < 0.01, then explain what this means biologically.

The analysis of the differential expression results for the airway smooth muscle (ASM) cells treated with dexamethasone (trt) versus untreated (untrt) shows the following:

*   **Number of significant genes at padj < 0.05:** 4,000
*   **Number of significant genes at padj < 0.01:** 2,901

### Biological Interpretation
The `padj` (adjusted p-value) represents the probability that a gene is identified as differentially expressed due to chance, corrected for multiple testing (e.g., using the Benjamini-Hochberg procedure). 

*   **Thresholding:** By lowering the threshold from 0.05 to 0.01, you are applying a more stringent criterion for statistical significance. This reduces the number of false positives (Type I errors) but increases the risk of false negatives (Type II errors), where truly differentially expressed genes might be excluded.
*   **Biological Context:** In RNA-seq studies, a `padj < 0.05` is a standard threshold for identifying a broad set of candidate genes. A `padj < 0.01` is often used when researchers want to focus on a more robust, high-confidence set of genes that are most likely to be biologically relevant to the treatment effect (in this case, the glucocorticoid response in ASM cells).

**Caveat:** Statistical significance does not always equate to biological significance. A gene with a very small `padj` might have a very small log2 fold change, meaning its expression change, while statistically reliable, may have a negligible impact on the cell's physiology. Always consider the magnitude of the `log2foldchange` alongside the `padj`.

tool_calls: [{'name': 'Database_Schema', 'args': {'input_text': 'dea_deseq2_results_airway_trt_vs_untrt'}, 'id': 'RoHZZgok', 'type': 'tool_call'}]
tool_calls: [{'name': 'SQL_Query', 'args': {'query': 'SELECT \n    (SELECT COUNT(*) FROM dea_deseq2_results_airway_trt_vs_untrt WHERE padj < 0.05) AS count_05,\n    (SELECT COUNT(*) FROM dea_deseq2_results_airway_trt_vs_untrt WHERE padj < 0.01) AS count_01;'}, 'id': 'aOg9fLFJ', 'type': 'tool_call'}]
tool_calls: [{'name': 'Retrieve_Context', 'args': {'query': 'padj significance threshold RNA-seq'}, 'id': 'WvFV9l0m', 'type': 'tool_call'}]
seconds: 13.179
prompt_tokens: 3524
completion_tokens: 366
total_tokens: 3890
reasoning_tokens: None
cost: None


## Router Agent

A router can choose whether a question should go to SQL tools, retrieval, or both. Here we still use the same real agent for the final work.


**Reflection Prompts**
- Check whether the router chooses the right evidence source for each question.
- Consider the cost of a wrong route: would the answer be incomplete, incorrect, or just slower?
- Discuss when routing should be handled by an LLM and when rules might be more appropriate.


In [48]:
questions = [
    "Which local tables are available?",
    "How many genes are significant at padj < 0.05 versus padj < 0.01?",
    "What is the DESeq2 result for CRISPLD2?",
    "What does the paper say about glucocorticoid response in airway smooth muscle?",
]

for question in questions:
    lower_question = question.lower()
    if "paper" in lower_question or "say about" in lower_question:
        expected_route = "retrieval_plus_answer"
    elif "tables" in lower_question or "available" in lower_question:
        expected_route = "schema"
    else:
        expected_route = "schema_plus_sql"

    chunks = []
    usage = {}
    start = time.perf_counter()
    markdown_text = chr(10).join(["### Question", question, "", "**Expected route:** " + expected_route, "", "_Streaming agent answer..._"])
    answer_display = display(Markdown(markdown_text), display_id=True)

    for event in rna_seq_agent.stream({"messages": [{"role": "user", "content": question}]}, stream_mode="values"):
        final_message = event["messages"][-1]
        tool_calls = getattr(final_message, "tool_calls", None)
        if tool_calls:
            print("tool_calls:", tool_calls)
        content = final_message.content
        answer_text = ""
        if isinstance(content, list):
            for block in content:
                if isinstance(block, dict) and block.get("type") == "text":
                    answer_text = answer_text + block.get("text", "") + chr(10) + chr(10)
        else:
            answer_text = str(content)
        chunks = [answer_text.strip()]
        markdown_text = chr(10).join(["### Question", question, "", "**Expected route:** " + expected_route, "", "### Agent answer", "".join(chunks)])
        answer_display.update(Markdown(markdown_text))
        if getattr(final_message, "usage_metadata", None):
            usage = final_message.usage_metadata

    output_token_details = usage.get("output_token_details", {}) if usage else {}
    print("seconds:", round(time.perf_counter() - start, 3))
    print("prompt_tokens:", usage.get("input_tokens"))
    print("completion_tokens:", usage.get("output_tokens"))
    print("total_tokens:", usage.get("total_tokens"))
    print("reasoning_tokens:", output_token_details.get("reasoning") or output_token_details.get("reasoning_tokens"))
    print("cost:", usage.get("cost"))
    display(Markdown("---"))


### Question
Which local tables are available?

**Expected route:** schema

### Agent answer
The following tables are available in the database:

*   **`airway_counts`**: Contains raw gene expression counts for various samples.
*   **`airway_metadata`**: Contains experimental metadata (e.g., cell type, dexamethasone treatment status).
*   **`dea_deseq2_results_airway_trt_vs_untrt`**: Contains differential expression analysis results comparing treated vs. untreated samples.
*   **`functional_gprofiler_ora_results`**: Contains functional enrichment results from g:Profiler.
*   **`functional_topgo_results_bp_all_de`**: Contains TopGO enrichment results for all differentially expressed genes (Biological Process).
*   **`functional_topgo_results_bp_downregulated`**: Contains TopGO enrichment results for downregulated genes (Biological Process).
*   **`functional_topgo_results_bp_upregulated`**: Contains TopGO enrichment results for upregulated genes (Biological Process).

**Caveat:** Ensure you use the appropriate table for your specific analysis (e.g., using `dea_deseq2_results_airway_trt_vs_untrt` for gene-level statistics and the `functional_` tables for pathway-level insights).

tool_calls: [{'name': 'Database_Schema', 'args': {}, 'id': 'QxjmfLD0', 'type': 'tool_call'}]
seconds: 45.06
prompt_tokens: 1288
completion_tokens: 268
total_tokens: 1556
reasoning_tokens: None
cost: None


---

### Question
How many genes are significant at padj < 0.05 versus padj < 0.01?

**Expected route:** schema_plus_sql

### Agent answer
There are **4,000** genes significant at an adjusted p-value (padj) < 0.05, and **2,901** genes significant at a more stringent padj < 0.01.

**Biological Interpretation:**
The higher number of significant genes at the 0.05 threshold indicates that relaxing the significance criteria captures a broader set of potentially differentially expressed genes, likely including those with smaller effect sizes or higher variability. The 2,901 genes at the 0.01 threshold represent a more confident set of genes with stronger evidence of differential expression between the treated and untreated conditions.

**Caveat:**
These counts are based on the Benjamini-Hochberg adjusted p-values (padj), which control the False Discovery Rate (FDR). While these thresholds are standard, the choice of threshold should ideally be balanced against the specific goals of the downstream analysis (e.g., prioritizing high-confidence candidates vs. capturing a comprehensive pathway signature).

tool_calls: [{'name': 'Database_Schema', 'args': {}, 'id': 'DJrtSy0u', 'type': 'tool_call'}]
tool_calls: [{'name': 'SQL_Query', 'args': {'query': 'SELECT \n    SUM(CASE WHEN padj < 0.05 THEN 1 ELSE 0 END) AS count_0_05,\n    SUM(CASE WHEN padj < 0.01 THEN 1 ELSE 0 END) AS count_0_01\nFROM dea_deseq2_results_airway_trt_vs_untrt;'}, 'id': 'gJskn31H', 'type': 'tool_call'}]
seconds: 5.429
prompt_tokens: 1454
completion_tokens: 210
total_tokens: 1664
reasoning_tokens: None
cost: None


...

tool_calls: [{'name': 'Database_Schema', 'args': {}, 'id': 'Taqg7AQe', 'type': 'tool_call'}]
tool_calls: [{'name': 'SQL_Query', 'args': {'query': "SELECT * FROM dea_deseq2_results_airway_trt_vs_untrt WHERE symbol = 'CRISPLD2'"}, 'id': 'xrAnZElU', 'type': 'tool_call'}]
seconds: 2.151
prompt_tokens: 1495
completion_tokens: 211
total_tokens: 1706
reasoning_tokens: None
cost: None


---

### Question
What does the paper say about glucocorticoid response in airway smooth muscle?

**Expected route:** retrieval_plus_answer

### Agent answer
The paper identifies **CRISPLD2** as a key glucocorticoid (GC)-responsive gene in airway smooth muscle (ASM) cells. The study highlights that while glucocorticoids are a mainstay therapy for asthma due to their anti-inflammatory effects, the specific mechanisms by which they act in ASM are not fully understood.

Key findings regarding the glucocorticoid response in ASM include:
*   **CRISPLD2 Induction:** Treatment with dexamethasone (DEX) significantly increases *CRISPLD2* mRNA expression in ASM cells.
*   **Immuno-modulation:** *CRISPLD2* acts as an inhibitory modulator of the immune response. When *CRISPLD2* is knocked down, ASM cells exhibit an enhanced inflammatory response (e.g., higher expression of *IL6* and *IL8*) when stimulated with the proinflammatory cytokine IL1β.
*   **Cell-Specific Effects:** The response to glucocorticoids can be cell-type specific. For example, while *CRISPLD2* expression is induced by DEX in ASM cells, it is observed to decrease in A549 pulmonary epithelial cells.
*   **Transcriptional Regulation:** The study suggests that the glucocorticoid receptor (GR) modulates gene expression in ASM, and it explores potential mechanisms involving transcription factors like CEBPD, which was also found to be significantly increased in response to DEX.

**Caveat:** The study notes that these findings are based on *in vitro* models of human ASM cells. The researchers emphasize that further studies are required to fully understand the cell-specific expression of *CRISPLD2* and its interactions with other asthma medications, as well as to confirm the specific transcriptional mechanisms (such as the role of CEBPB binding) in a clinical or *in vivo* context.

tool_calls: [{'name': 'Retrieve_Context', 'args': {'query': 'glucocorticoid response in airway smooth muscle'}, 'id': 'km0FbHH8', 'type': 'tool_call'}]
seconds: 2.216
prompt_tokens: 2623
completion_tokens: 374
total_tokens: 2997
reasoning_tokens: None
cost: None


---

## Sequential Chain

A sequential chain is more deterministic: ask the agent focused subquestions, then ask it to combine the answers.


**Reflection Prompts**
- Compare the sequential chain with the router agent: which is more predictable, and which is more flexible?
- Evaluate whether breaking the task into focused subquestions improves the final synthesis.
- Look for information loss between intermediate answers and the final combined answer.


In [49]:
subquestions = [
    "How many samples are in each dex treatment group?",
    "How many genes are significant at padj < 0.05 versus padj < 0.01?",
    "Which 8 genes have the largest positive log2FoldChange among genes with padj < 0.01?",
    "Which topGO biological processes are enriched among upregulated genes?",
]

subanswers = ""
for subquestion in subquestions:
    chunks = []
    usage = {}
    start = time.perf_counter()
    markdown_text = chr(10).join(["### Subquestion", subquestion, "", "_Streaming agent answer..._"])
    answer_display = display(Markdown(markdown_text), display_id=True)

    for event in rna_seq_agent.stream({"messages": [{"role": "user", "content": subquestion}]}, stream_mode="values"):
        final_message = event["messages"][-1]
        tool_calls = getattr(final_message, "tool_calls", None)
        if tool_calls:
            print("tool_calls:", tool_calls)
        content = final_message.content
        answer_text = ""
        if isinstance(content, list):
            for block in content:
                if isinstance(block, dict) and block.get("type") == "text":
                    answer_text = answer_text + block.get("text", "") + chr(10) + chr(10)
        else:
            answer_text = str(content)
        chunks = [answer_text.strip()]
        markdown_text = chr(10).join(["### Subquestion", subquestion, "", "### Agent answer", "".join(chunks)])
        answer_display.update(Markdown(markdown_text))
        if getattr(final_message, "usage_metadata", None):
            usage = final_message.usage_metadata

    subanswers = subanswers + chr(10) + chr(10) + "Question: " + subquestion + chr(10) + "Answer:" + chr(10) + "".join(chunks).strip()
    output_token_details = usage.get("output_token_details", {}) if usage else {}
    print("seconds:", round(time.perf_counter() - start, 3))
    print("prompt_tokens:", usage.get("input_tokens"))
    print("completion_tokens:", usage.get("output_tokens"))
    print("total_tokens:", usage.get("total_tokens"))
    print("reasoning_tokens:", output_token_details.get("reasoning") or output_token_details.get("reasoning_tokens"))
    print("cost:", usage.get("cost"))

final_question = "Summarize these subanswers as a compact teaching answer. Separate exact SQL-derived facts from biological interpretation."
chunks = []
usage = {}
start = time.perf_counter()
markdown_text = chr(10).join(["### Final answer", "", "_Streaming agent answer..._"])
answer_display = display(Markdown(markdown_text), display_id=True)

for event in rna_seq_agent.stream({"messages": [{"role": "user", "content": final_question + subanswers}]}, stream_mode="values"):
    final_message = event["messages"][-1]
    tool_calls = getattr(final_message, "tool_calls", None)
    if tool_calls:
        print("tool_calls:", tool_calls)
    content = final_message.content
    answer_text = ""
    if isinstance(content, list):
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                answer_text = answer_text + block.get("text", "") + chr(10) + chr(10)
    else:
        answer_text = str(content)
    chunks = [answer_text.strip()]
    markdown_text = chr(10).join(["### Final answer", "", "".join(chunks)])
    answer_display.update(Markdown(markdown_text))
    if getattr(final_message, "usage_metadata", None):
        usage = final_message.usage_metadata

output_token_details = usage.get("output_token_details", {}) if usage else {}
print("seconds:", round(time.perf_counter() - start, 3))
print("prompt_tokens:", usage.get("input_tokens"))
print("completion_tokens:", usage.get("output_tokens"))
print("total_tokens:", usage.get("total_tokens"))
print("reasoning_tokens:", output_token_details.get("reasoning") or output_token_details.get("reasoning_tokens"))
print("cost:", usage.get("cost"))


### Subquestion
How many samples are in each dex treatment group?

_Streaming agent answer..._

tool_calls: [{'name': 'Database_Schema', 'args': {'input_text': 'airway_metadata'}, 'id': 'eraD0pXO', 'type': 'tool_call'}]
tool_calls: [{'name': 'Sample_Column_Values', 'args': {'input_text': 'airway_metadata.dex'}, 'id': 'OPIgKHfr', 'type': 'tool_call'}]
tool_calls: [{'name': 'SQL_Query', 'args': {'query': 'SELECT dex, COUNT(*) as sample_count FROM airway_metadata GROUP BY dex;'}, 'id': 'eYg8wqzl', 'type': 'tool_call'}]
seconds: 2.946
prompt_tokens: 3188
completion_tokens: 123
total_tokens: 3311
reasoning_tokens: None
cost: None


### Subquestion
How many genes are significant at padj < 0.05 versus padj < 0.01?

### Agent answer
There are **4,000** genes significant at an adjusted p-value (padj) < 0.05, and **2,901** genes significant at a more stringent padj < 0.01.

**Biological Interpretation:**
The higher number of significant genes at the 0.05 threshold indicates that relaxing the significance criteria captures a broader set of potentially differentially expressed genes, likely including those with smaller effect sizes or higher variability. The 2,901 genes at the 0.01 threshold represent a more confident set of genes with stronger evidence of differential expression between the treated and untreated conditions.

**Caveat:**
These counts are based on the Benjamini-Hochberg adjusted p-values (padj), which control the False Discovery Rate (FDR). While these thresholds are standard, the choice of threshold should ideally be balanced against the specific goals of the downstream analysis (e.g., prioritizing high-confidence candidates vs. capturing a comprehensive pathway signature).

tool_calls: [{'name': 'Database_Schema', 'args': {}, 'id': 'UtNRnwrZ', 'type': 'tool_call'}]
tool_calls: [{'name': 'SQL_Query', 'args': {'query': 'SELECT \n    SUM(CASE WHEN padj < 0.05 THEN 1 ELSE 0 END) AS count_0_05,\n    SUM(CASE WHEN padj < 0.01 THEN 1 ELSE 0 END) AS count_0_01\nFROM dea_deseq2_results_airway_trt_vs_untrt;'}, 'id': 'UJorhtKt', 'type': 'tool_call'}]
seconds: 3.431
prompt_tokens: 1454
completion_tokens: 210
total_tokens: 1664
reasoning_tokens: None
cost: None


### Subquestion
Which 8 genes have the largest positive log2FoldChange among genes with padj < 0.01?

### Agent answer
The 8 genes with the largest positive log2FoldChange (indicating upregulation in the treated group) among those with an adjusted p-value (padj) < 0.01 are:

| Symbol | log2FoldChange | padj |
| :--- | :--- | :--- |
| ALOX15B | 9.51 | 1.18e-17 |
| ZBTB16 | 7.35 | 2.32e-40 |
| RP11-357D18.1 | 6.33 | 6.03e-19 |
| GUCY2D | 5.88 | 8.86e-05 |
| STEAP4 | 5.21 | 5.06e-24 |
| RP11-434D9.1 | 5.09 | 1.13e-04 |
| ANGPTL7 | 5.08 | 5.18e-10 |
| PRODH | 4.89 | 4.09e-19 |

**Biological Interpretation:** These genes show the strongest evidence of upregulation following treatment. For example, *ALOX15B* (Arachidonate 15-Lipoxygenase B) and *ZBTB16* are known to be highly responsive to glucocorticoid treatment in airway smooth muscle cells.

**Caveat:** While these genes have the highest fold changes, high fold change does not always equate to high biological significance; it is important to consider the base expression levels (baseMean) and the specific biological context of the experiment.

tool_calls: [{'name': 'Database_Schema', 'args': {}, 'id': 'BHPBPnY9', 'type': 'tool_call'}]
tool_calls: [{'name': 'SQL_Query', 'args': {'query': 'SELECT symbol, log2foldchange, padj FROM dea_deseq2_results_airway_trt_vs_untrt WHERE padj < 0.01 ORDER BY log2foldchange DESC LIMIT 8;'}, 'id': '6kQM4l9l', 'type': 'tool_call'}]
seconds: 39.711
prompt_tokens: 1644
completion_tokens: 373
total_tokens: 2017
reasoning_tokens: None
cost: None


### Subquestion
Which topGO biological processes are enriched among upregulated genes?

### Agent answer
The top biological processes enriched among upregulated genes in the airway dataset (based on the `functional_topgo_results_bp_upregulated` table) include **cell adhesion**, **angiogenesis**, and **positive regulation of cell migration**.

### Top 5 Enriched Biological Processes (by `weight01` p-value):

| GO ID | Term | Significant Genes | weight01 (p-value) |
| :--- | :--- | :--- | :--- |
| GO:0007155 | cell adhesion | 139 | 0.000002 |
| GO:0001525 | angiogenesis | 64 | 0.000009 |
| GO:0030335 | positive regulation of cell migration | 59 | 0.000011 |
| GO:0001937 | negative regulation of endothelial cell proliferation | 10 | 0.000024 |
| GO:0044320 | cellular response to leptin stimulus | 7 | 0.000100 |

**Biological Interpretation:**
The enrichment of terms like "cell adhesion," "angiogenesis," and "cell migration" suggests that dexamethasone treatment in these airway smooth muscle cells significantly alters structural and migratory pathways, potentially impacting tissue remodeling or inflammatory responses.

**Caveat:**
The `weight01` algorithm in TopGO is designed to account for the hierarchical structure of Gene Ontology terms to reduce redundancy, but these results represent statistical enrichment and do not confirm functional changes without further experimental validation.

tool_calls: [{'name': 'Database_Schema', 'args': {}, 'id': 'kC6FEV5z', 'type': 'tool_call'}]
tool_calls: [{'name': 'SQL_Query', 'args': {'query': 'SELECT * FROM functional_topgo_results_bp_upregulated ORDER BY weight01 ASC LIMIT 10;'}, 'id': 'yszPUdip', 'type': 'tool_call'}]
seconds: 3.276
prompt_tokens: 1741
completion_tokens: 355
total_tokens: 2096
reasoning_tokens: None
cost: None


### Final answer

### Summary of Airway RNA-seq Analysis

#### **Experimental Design & Statistical Findings**
*   **Study Design:** The experiment is balanced with 4 biological replicates per group (`trt` vs. `untrt`).
*   **Differential Expression:** There are 4,000 genes significant at $padj < 0.05$ and 2,901 genes at $padj < 0.01$.
*   **Top Upregulated Genes ($padj < 0.01$):** The most strongly upregulated genes include *ALOX15B* (LFC: 9.51), *ZBTB16* (LFC: 7.35), and *RP11-357D18.1* (LFC: 6.33).
*   **Functional Enrichment:** Upregulated genes are significantly enriched for biological processes including **cell adhesion**, **angiogenesis**, and **positive regulation of cell migration**.

#### **Biological Interpretation**
The balanced design provides robust statistical power for detecting differential expression. The high number of significant genes and the enrichment of structural/migratory pathways (e.g., cell adhesion, angiogenesis) suggest that dexamethasone induces substantial tissue remodeling and functional changes in airway smooth muscle cells. Genes like *ALOX15B* and *ZBTB16* serve as strong markers of this glucocorticoid response.

#### **Caveats**
*   **Statistical:** While FDR-adjusted p-values control for false positives, the choice of threshold should align with the study's goal (e.g., discovery vs. validation).
*   **Biological:** High log2FoldChange does not guarantee biological relevance; base expression levels should be considered. Furthermore, enrichment results indicate statistical over-representation and require experimental validation to confirm functional impact.
*   **Methodological:** Replicates should be analyzed using paired models to account for potential batch effects or inter-individual variability inherent in the cell lines.

seconds: 2.049
prompt_tokens: 1790
completion_tokens: 421
total_tokens: 2211
reasoning_tokens: None
cost: None


## 9. Memory Techniques

Conversation Memory remembers everything; Windowed Memory remembers only the recent part of the conversation to save time and tokens.

```text
Conversation Memory

Past                                              Present
│                                                   │
▼                                                   ▼
[1]─[2]─[3]─[4]─[5]─[6]─[7]─[8]─[9]─[10]
════════════════════════════════════════
          All messages are remembered.


Windowed Memory (window = 4)

Past                                              Present
│                                                   │
▼                                                   ▼
[1]─[2]─[3]─[4]─[5]─[6]─[7]─[8]─[9]─[10]
                        ════════════════
                     Only recent messages
```

**Reflection Prompts**
- Compare how memory changes the model behavior across follow-up questions.
- Distinguish useful conversational context from irrelevant history that could distract the model.
- Think about what should be remembered explicitly in a scientific workflow and what should be recomputed from data.


### i. No Memory

Each model call is independent.


**Reflection Prompts**
- Observe which follow-up questions become ambiguous when previous messages are not included.
- Identify whether the model admits missing context or fills the gap by guessing.
- Discuss when independent calls are preferable for reproducibility and evaluation.


In [50]:
prompts = [
    {"turn": "first", "prompt": "The local dataset is the airway RNA-seq dataset comparing dexamethasone-treated and untreated human airway smooth muscle cell samples."},
    {"turn": "followup_without_memory", "prompt": "What treatment comparison is the local dataset about?"},
]

for spec in prompts:
    chunks = []
    usage = {}
    start = time.perf_counter()
    print(f"### {spec['turn']}\n")

    for chunk in model.stream(spec["prompt"]):
        if chunk.content:
            text = chunk.text()
            chunks.append(text)
            print(text, end="")
        if chunk.usage_metadata:
            usage = chunk.usage_metadata

    answer = "".join(chunks)
    output_token_details = usage.get("output_token_details", {}) if usage else {}
    print("\n\nseconds:", round(time.perf_counter() - start, 3))
    print("prompt_tokens:", usage.get("input_tokens"))
    print("completion_tokens:", usage.get("output_tokens"))
    print("total_tokens:", usage.get("total_tokens"))
    print("reasoning_tokens:", output_token_details.get("reasoning") or output_token_details.get("reasoning_tokens"))
    print("cost:", usage.get("cost"))


### first



/opt/miniconda3/envs/agent_teaching_py312/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  exec(code_obj, self.user_global_ns, self.user_ns)


To provide you with the most relevant analysis or guidance for your **airway RNA-seq dataset (dexamethasone-treated vs. untreated human airway smooth muscle cells)**, I have outlined the standard bioinformatics workflow and key biological considerations specific to this experimental design.

### 1. Contextualizing the Dataset
Dexamethasone (a synthetic glucocorticoid) is a potent anti-inflammatory drug. In airway smooth muscle (ASM) cells, it is known to:
*   **Inhibit pro-inflammatory cytokines** (e.g., IL-6, CXCL8).
*   **Induce anti-inflammatory genes** (e.g., DUSP1, GILZ/TSC22D3).
*   **Alter structural remodeling genes** (e.g., extracellular matrix proteins).

### 2. Recommended Analysis Workflow
If you are currently processing this data, here is the standard pipeline:

*   **Quality Control (QC):** Use `FastQC` and `MultiQC` to check for adapter contamination and base quality.
*   **Alignment:** Map reads to the human reference genome (GRCh38) using `STAR` or `HISAT2`.
*   **Quan

/opt/miniconda3/envs/agent_teaching_py312/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  exec(code_obj, self.user_global_ns, self.user_ns)


To provide you with an accurate answer, **I need you to share the dataset or describe its contents.**

As an AI, I do not have access to your local files or private computer storage unless you upload the file to this chat or paste the relevant information here.

**Once you provide the data, I can help you identify:**
1.  **The Treatment Groups:** (e.g., Drug A vs. Placebo, Intervention vs. Control).
2.  **The Outcome Measures:** (e.g., blood pressure, test scores, recovery time).
3.  **The Statistical Comparison:** (e.g., t-tests, ANOVA, or regression models used to compare the groups).

**How to share it:**
*   **Upload the file:** Click the paperclip or "plus" icon to upload a CSV, Excel, or text file.
*   **Paste a sample:** Copy and paste the first few rows (the header and a few data points) into the chat.
*   **Describe the columns:** Tell me the names of the columns and what the values represent.

**Please upload or paste the information, and I will analyze it for you immediately

### ii. Conversation Memory

Now the follow-up question is sent together with earlier messages.


**Reflection Prompts**
- Compare the answer with the no-memory version: what information is recovered from earlier turns?
- Check whether memory improves continuity without introducing unsupported assumptions.
- Consider how long conversation history affects cost, privacy, and reproducibility.


In [51]:
messages = [
    {"role": "user", "content": "The local dataset compares dexamethasone-treated and untreated airway smooth muscle cell samples."},
    {"role": "assistant", "content": "Noted."},
    {"role": "user", "content": "What treatment comparison is the local dataset about?"},
]
chunks = []
usage = {}
start = time.perf_counter()
for chunk in model.stream(messages):
    if chunk.content:
        text = chunk.text()
        chunks.append(text)
        print(text, end="")
    if chunk.usage_metadata:
        usage = chunk.usage_metadata

answer = "".join(chunks)
output_token_details = usage.get("output_token_details", {}) if usage else {}
print("\n\nseconds:", round(time.perf_counter() - start, 3))
print("prompt_tokens:", usage.get("input_tokens"))
print("completion_tokens:", usage.get("output_tokens"))
print("total_tokens:", usage.get("total_tokens"))
print("reasoning_tokens:", output_token_details.get("reasoning") or output_token_details.get("reasoning_tokens"))
print("cost:", usage.get("cost"))


The local dataset compares **dexamethasone-treated** airway smooth muscle cells against **untreated** (control) airway smooth muscle cells.

seconds: 0.408
prompt_tokens: 0
completion_tokens: 0
total_tokens: 0
reasoning_tokens: None
cost: None


/opt/miniconda3/envs/agent_teaching_py312/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  exec(code_obj, self.user_global_ns, self.user_ns)


### iii. Windowed Memory

Windowed memory keeps only recent messages. It can lose useful earlier details.


**Reflection Prompts**
- Observe which details survive when only recent messages are kept.
- Evaluate whether windowed memory preserves enough context for the follow-up question.
- Discuss how you would choose a memory window size for a real analysis assistant.


In [52]:
full_history = [
    {"role": "user", "content": "The local files include airway_counts.csv, airway_metadata.csv, data/dea/DESeq2_results_airway_trt_vs_untrt.csv, and functional enrichment TXT files from gProfiler and topGO."},
    {"role": "assistant", "content": "Noted."},
    {"role": "user", "content": "The key treatment variable is the dex column with trt and untrt groups, and enrichment files help interpret gene sets rather than measure expression directly."},
    {"role": "assistant", "content": "Noted."},
    {"role": "user", "content": "Which local files define the treatment comparison, measured gene expression, differential expression, and functional interpretation?"},
]

for spec in [
    {"memory": "short_window", "messages": full_history[-3:]},
    {"memory": "full_conversation", "messages": full_history},
]:
    chunks = []
    usage = {}
    start = time.perf_counter()
    print(f"### {spec['memory']}\n")

    for chunk in model.stream(spec["messages"]):
        if chunk.content:
            text = chunk.text()
            chunks.append(text)
            print(text, end="")
        if chunk.usage_metadata:
            usage = chunk.usage_metadata

    answer = "".join(chunks)
    output_token_details = usage.get("output_token_details", {}) if usage else {}
    print("\n\nseconds:", round(time.perf_counter() - start, 3))
    print("prompt_tokens:", usage.get("input_tokens"))
    print("completion_tokens:", usage.get("output_tokens"))
    print("total_tokens:", usage.get("total_tokens"))
    print("reasoning_tokens:", output_token_details.get("reasoning") or output_token_details.get("reasoning_tokens"))
    print("cost:", usage.get("cost"))


### short_window



/opt/miniconda3/envs/agent_teaching_py312/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  exec(code_obj, self.user_global_ns, self.user_ns)


To provide a precise answer, I would need to see the file list in your current working directory. However, based on standard bioinformatics workflows (such as those using DESeq2, edgeR, or GSEA), here is how you can identify these files:

### 1. Treatment Comparison (Metadata/Design)
*   **What to look for:** Files ending in `.csv`, `.tsv`, or `.txt` that contain sample IDs and a column named `dex` (or similar).
*   **Common names:** `metadata.csv`, `samples.txt`, `design_matrix.csv`, or `coldata.csv`.
*   **Purpose:** This file maps your sample names to the `trt` and `untrt` groups. It is the "key" that tells the software which samples belong to which experimental condition.

### 2. Measured Gene Expression (Raw Counts)
*   **What to look for:** Large matrix files where rows are genes and columns are samples.
*   **Common names:** `counts.csv`, `raw_counts.txt`, `gene_expression_matrix.tsv`, or output files from quantification tools like `featureCounts` or `Salmon`.
*   **Purpose:** T

/opt/miniconda3/envs/agent_teaching_py312/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  exec(code_obj, self.user_global_ns, self.user_ns)


Based on the files provided, here is the breakdown of their roles in your analysis:

### 1. Treatment Comparison
*   **`airway_metadata.csv`**: This file defines the experimental design. It contains the `dex` column (with `trt` and `untrt` groups), which maps the samples to their respective treatment conditions. This is the "ground truth" for your experimental groups.

### 2. Measured Gene Expression
*   **`airway_counts.csv`**: This file contains the raw (or normalized) read counts for each gene across all samples. It represents the actual measured expression levels before any statistical testing for differential expression was performed.

### 3. Differential Expression
*   **`data/dea/DESeq2_results_airway_trt_vs_untrt.csv`**: This file contains the statistical output of the comparison between the `trt` and `untrt` groups. It typically includes log2 fold changes, p-values, and adjusted p-values (padj) for each gene, identifying which genes are significantly up- or down-regulated due 

## 10. Tool Usage


**Reflection Prompts**
- Compare answers with no tools, one tool, and multiple tools in terms of evidence quality.
- Identify which tasks need computation, which need retrieval, and which need both.
- Ask how tool access changes the model from a text generator into an analysis workflow coordinator.


### i. No Tools

The model gets no local table values.


**Reflection Prompts**
- Check which parts of the answer are plausible but unsupported by local data.
- Compare confidence, specificity, and correctness with later tool-based answers.
- Discuss when a no-tool answer is acceptable and when it is scientifically risky.


In [53]:
question = "What is the DESeq2 log2FoldChange and adjusted p-value for CRISPLD2, and which upregulated biological processes are enriched in the local airway dataset?"
chunks = []
usage = {}
start = time.perf_counter()
for chunk in model.stream(question):
    if chunk.content:
        text = chunk.text()
        chunks.append(text)
        print(text, end="")
    if chunk.usage_metadata:
        usage = chunk.usage_metadata

answer = "".join(chunks)
output_token_details = usage.get("output_token_details", {}) if usage else {}
print("\n\nseconds:", round(time.perf_counter() - start, 3))
print("prompt_tokens:", usage.get("input_tokens"))
print("completion_tokens:", usage.get("output_tokens"))
print("total_tokens:", usage.get("total_tokens"))
print("reasoning_tokens:", output_token_details.get("reasoning") or output_token_details.get("reasoning_tokens"))
print("cost:", usage.get("cost"))


/opt/miniconda3/envs/agent_teaching_py312/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  exec(code_obj, self.user_global_ns, self.user_ns)


To provide the specific values for *CRISPLD2* and the enriched biological processes, I am referencing the landmark study **"Host-viral infection maps reveal signatures of severe COVID-19 patients"** (published in *Cell*, 2020, by Blanco-Melo et al.), which is the primary dataset containing local airway (nasopharyngeal/tracheal) transcriptomic data for COVID-19.

### 1. *CRISPLD2* Statistics (Local Airway Dataset)
In the context of SARS-CoV-2 infection in the airway epithelium, *CRISPLD2* is consistently identified as a significantly upregulated gene. Based on the differential expression analysis (DESeq2) from the Blanco-Melo et al. dataset:

*   **log2FoldChange:** Approximately **2.5 to 3.2** (depending on the specific cohort subset, such as the NHBE cell line vs. patient samples).
*   **Adjusted p-value (padj):** Highly significant, typically **< 1.0e-10**.

*Note: CRISPLD2 is a known glucocorticoid-responsive gene and is often upregulated in the airway as part of the host's attempt 

### ii. Python Tool

Plain pandas code can calculate exact summaries.


**Reflection Prompts**
- Compare the Python result with the model-only answer: what becomes exact rather than guessed?
- Identify which operations are easier or clearer in pandas than in natural language reasoning.
- Consider how code execution changes the standard for checking the final answer.


In [54]:
question = 'What is the DESeq2 result for CRISPLD2?'
chunks = []
usage = {}
start = time.perf_counter()
markdown_text = chr(10).join(["### Question", question, "", "_Streaming agent answer..._"])
answer_display = display(Markdown(markdown_text), display_id=True)

for event in rna_seq_agent.stream({"messages": [{"role": "user", "content": question}]}, stream_mode="values"):
    final_message = event["messages"][-1]
    tool_calls = getattr(final_message, "tool_calls", None)
    if tool_calls:
        print("tool_calls:", tool_calls)
    content = final_message.content
    answer_text = ""
    if isinstance(content, list):
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                answer_text = answer_text + block.get("text", "") + "\n\n"
            else:
                answer_text = answer_text + str(block) + "\n\n"
    else:
        answer_text = str(content)
    chunks = [answer_text.strip()]
    markdown_text = chr(10).join(["### Question", question, "", "".join(chunks)])
    answer_display.update(Markdown(markdown_text))
    if getattr(final_message, "usage_metadata", None):
        usage = final_message.usage_metadata

output_token_details = usage.get("output_token_details", {}) if usage else {}
print("seconds:", round(time.perf_counter() - start, 3))
print("prompt_tokens:", usage.get("input_tokens"))
print("completion_tokens:", usage.get("output_tokens"))
print("total_tokens:", usage.get("total_tokens"))
print("reasoning_tokens:", output_token_details.get("reasoning") or output_token_details.get("reasoning_tokens"))
print("cost:", usage.get("cost"))


### Question
What is the DESeq2 result for CRISPLD2?

The DESeq2 results for the gene **CRISPLD2** in the airway treatment vs. untreated comparison are as follows:

*   **log2FoldChange:** 2.63
*   **p-value:** 2.45e-48
*   **padj (Adjusted p-value):** 7.88e-46
*   **baseMean:** 3082.09

**Biological Interpretation:**
CRISPLD2 is significantly upregulated in the treated airway smooth muscle cells, with a large positive log2 fold change and a highly significant adjusted p-value. This gene is known to be glucocorticoid-responsive and is often associated with anti-inflammatory pathways in airway tissues.

**Caveat:**
While the statistical significance is very high, these results are based on the specific experimental conditions (dexamethasone treatment of airway smooth muscle cells) provided in the dataset. Results may vary in different cell types or under different experimental conditions.

tool_calls: [{'name': 'Database_Schema', 'args': {}, 'id': 'vbbNX38L', 'type': 'tool_call'}]
tool_calls: [{'name': 'SQL_Query', 'args': {'query': "SELECT * FROM dea_deseq2_results_airway_trt_vs_untrt WHERE symbol = 'CRISPLD2'"}, 'id': '1DyoxjbL', 'type': 'tool_call'}]
seconds: 2.691
prompt_tokens: 1495
completion_tokens: 211
total_tokens: 1706
reasoning_tokens: None
cost: None


### iii. SQL Tool

SQL is useful for exact structured questions.


**Reflection Prompts**
- Evaluate whether SQL is the right abstraction for the structured question being asked.
- Check if the generated query matches the biological intent of the natural-language question.
- Discuss how schema inspection and sample values reduce the risk of querying the wrong column or category.


In [55]:
question = 'How many genes are significant at padj < 0.01?'
chunks = []
usage = {}
start = time.perf_counter()
markdown_text = chr(10).join(["### Question", question, "", "_Streaming agent answer..._"])
answer_display = display(Markdown(markdown_text), display_id=True)

for event in rna_seq_agent.stream({"messages": [{"role": "user", "content": question}]}, stream_mode="values"):
    final_message = event["messages"][-1]
    tool_calls = getattr(final_message, "tool_calls", None)
    if tool_calls:
        print("tool_calls:", tool_calls)
    content = final_message.content
    answer_text = ""
    if isinstance(content, list):
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                answer_text = answer_text + block.get("text", "") + "\n\n"
            else:
                answer_text = answer_text + str(block) + "\n\n"
    else:
        answer_text = str(content)
    chunks = [answer_text.strip()]
    markdown_text = chr(10).join(["### Question", question, "", "".join(chunks)])
    answer_display.update(Markdown(markdown_text))
    if getattr(final_message, "usage_metadata", None):
        usage = final_message.usage_metadata

output_token_details = usage.get("output_token_details", {}) if usage else {}
print("seconds:", round(time.perf_counter() - start, 3))
print("prompt_tokens:", usage.get("input_tokens"))
print("completion_tokens:", usage.get("output_tokens"))
print("total_tokens:", usage.get("total_tokens"))
print("reasoning_tokens:", output_token_details.get("reasoning") or output_token_details.get("reasoning_tokens"))
print("cost:", usage.get("cost"))


### Question
How many genes are significant at padj < 0.01?

There are 2,901 genes that are statistically significant with an adjusted p-value (padj) of less than 0.01 in the airway treatment vs. untreated comparison.

**Biological Interpretation:**
This result indicates that a substantial portion of the transcriptome is differentially expressed in response to the treatment (dexamethasone), suggesting a broad regulatory impact on the airway smooth muscle cells.

**Caveat:**
While these genes meet the statistical threshold for significance, the biological relevance of each gene depends on the magnitude of the log2 fold change and the specific experimental context. Statistical significance does not always equate to biological significance.

tool_calls: [{'name': 'Database_Schema', 'args': {'input_text': 'dea_deseq2_results_airway_trt_vs_untrt'}, 'id': 'ZPVDlQmR', 'type': 'tool_call'}]
tool_calls: [{'name': 'SQL_Query', 'args': {'query': 'SELECT count(*) FROM dea_deseq2_results_airway_trt_vs_untrt WHERE padj < 0.01'}, 'id': 'ghvVNbkT', 'type': 'tool_call'}]
seconds: 11.403
prompt_tokens: 1402
completion_tokens: 130
total_tokens: 1532
reasoning_tokens: None
cost: None


### iv. Multiple Tools

The strongest answers combine several local evidence sources.


**Reflection Prompts**
- Trace how different tools contribute complementary evidence to the final answer.
- Look for conflicts between sources and decide how the agent should handle them.
- Evaluate whether multiple tools improve completeness enough to justify added complexity and latency.


In [56]:
question = 'Combine CRISPLD2 differential expression, significant gene counts at padj < 0.05 versus 0.01, and upregulated topGO enrichment terms.'
chunks = []
usage = {}
start = time.perf_counter()
markdown_text = chr(10).join(["### Question", question, "", "_Streaming agent answer..._"])
answer_display = display(Markdown(markdown_text), display_id=True)

for event in rna_seq_agent.stream({"messages": [{"role": "user", "content": question}]}, stream_mode="values"):
    final_message = event["messages"][-1]
    tool_calls = getattr(final_message, "tool_calls", None)
    if tool_calls:
        print("tool_calls:", tool_calls)
    content = final_message.content
    answer_text = ""
    if isinstance(content, list):
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                answer_text = answer_text + block.get("text", "") + "\n\n"
            else:
                answer_text = answer_text + str(block) + "\n\n"
    else:
        answer_text = str(content)
    chunks = [answer_text.strip()]
    markdown_text = chr(10).join(["### Question", question, "", "".join(chunks)])
    answer_display.update(Markdown(markdown_text))
    if getattr(final_message, "usage_metadata", None):
        usage = final_message.usage_metadata

output_token_details = usage.get("output_token_details", {}) if usage else {}
print("seconds:", round(time.perf_counter() - start, 3))
print("prompt_tokens:", usage.get("input_tokens"))
print("completion_tokens:", usage.get("output_tokens"))
print("total_tokens:", usage.get("total_tokens"))
print("reasoning_tokens:", output_token_details.get("reasoning") or output_token_details.get("reasoning_tokens"))
print("cost:", usage.get("cost"))


### Question
Combine CRISPLD2 differential expression, significant gene counts at padj < 0.05 versus 0.01, and upregulated topGO enrichment terms.

The gene **CRISPLD2** is significantly upregulated in the treated airway samples, with a log2 fold change of approximately **2.63** and a highly significant adjusted p-value (**padj ≈ 7.88e-46**).

Regarding the global differential expression results:
*   There are **4,000** genes with an adjusted p-value < 0.05.
*   There are **2,901** genes with an adjusted p-value < 0.01.

The top upregulated biological process (BP) enrichment terms (based on TopGO weight01) include:
1.  **Cell adhesion** (139 significant genes)
2.  **Angiogenesis** (64 significant genes)
3.  **Positive regulation of cell migration** (59 significant genes)

**Biological Interpretation:**
CRISPLD2 is a known glucocorticoid-responsive gene in airway smooth muscle cells, often associated with anti-inflammatory responses and tissue remodeling. The enrichment of terms like "cell adhesion" and "angiogenesis" suggests that dexamethasone treatment significantly alters the structural and migratory properties of these cells.

**Caveat:**
The gene counts and enrichment results are based on the specific statistical thresholds and annotation databases used in this analysis. Enrichment results represent statistical associations and do not necessarily imply direct functional causality for all genes within those categories.

tool_calls: [{'name': 'Database_Schema', 'args': {}, 'id': 'XC4Eg8BG', 'type': 'tool_call'}]
tool_calls: [{'name': 'SQL_Query', 'args': {'query': "SELECT * FROM dea_deseq2_results_airway_trt_vs_untrt WHERE symbol = 'CRISPLD2'"}, 'id': 'Pt592jNF', 'type': 'tool_call'}]
tool_calls: [{'name': 'SQL_Query', 'args': {'query': 'SELECT \n    (SELECT COUNT(*) FROM dea_deseq2_results_airway_trt_vs_untrt WHERE padj < 0.05) AS count_0_05,\n    (SELECT COUNT(*) FROM dea_deseq2_results_airway_trt_vs_untrt WHERE padj < 0.01) AS count_0_01'}, 'id': 'B7FOKR83', 'type': 'tool_call'}]
tool_calls: [{'name': 'SQL_Query', 'args': {'query': 'SELECT term, significant, weight01 FROM functional_topgo_results_bp_upregulated ORDER BY weight01 ASC LIMIT 5'}, 'id': 'qFHUsNxG', 'type': 'tool_call'}]
seconds: 21.615
prompt_tokens: 1839
completion_tokens: 297
total_tokens: 2136
reasoning_tokens: None
cost: None


### v. Tool Descriptions

Clear descriptions help an agent choose the right evidence source. Here we show the same idea with simple labels.


**Reflection Prompts**
- Compare how different tool descriptions might influence the agent selection process.
- Identify descriptions that are too vague, too narrow, or likely to cause the wrong tool to be chosen.
- Rewrite one tool description in your own words to make its intended use clearer.


In [57]:
tool_descriptions = pd.DataFrame([
    {
        "tool_name": "Database_Schema",
        "clear_description": "Use first for data questions. It tells the agent which tables and columns exist before SQL is written.",
        "poor_description": "Show data info.",
    },
    {
        "tool_name": "Sample_Column_Values",
        "clear_description": "Use before WHERE filters on text columns. It prevents guessing values such as gene symbols, treatment labels, or GO terms.",
        "poor_description": "Show examples.",
    },
    {
        "tool_name": "SQL_Query",
        "clear_description": "Run the agent-generated read-only SELECT query and return actual rows. This is where exact numerical answers come from.",
        "poor_description": "Search gene stuff.",
    },
    {
        "tool_name": "Retrieve_Context",
        "clear_description": "Retrieve paper or file context when the question asks for interpretation or provenance beyond exact table values.",
        "poor_description": "Search text.",
    },
])

df = tool_descriptions
df.style.set_properties(
    subset=["clear_description", "poor_description"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


,tool_name,clear_description,poor_description
0,Database_Schema,Use first for data questions. It tells the agent which tables and columns exist before SQL is written.,Show data info.
1,Sample_Column_Values,"Use before WHERE filters on text columns. It prevents guessing values such as gene symbols, treatment labels, or GO terms.",Show examples.
2,SQL_Query,Run the agent-generated read-only SELECT query and return actual rows. This is where exact numerical answers come from.,Search gene stuff.
3,Retrieve_Context,Retrieve paper or file context when the question asks for interpretation or provenance beyond exact table values.,Search text.


## Open ended exercise

Choose one small change you would like to make to the agent tools.

You can either:
- change a fixed plotting choice, such as the volcano plot colors, title, figure size, or axis labels;
- change a configurable tool argument, such as `padj_cutoff`, `lfc_cutoff`, or `top_n_labels`;
- write a new natural-language question that asks the agent to use the tool differently.

Run the agent before and after your change, and briefly reflect on what changed in the output.

In [ ]:
QUESTION = "Plot a volcano plot with padj < 0.01 and label the top 5 genes."

chunks = []
usage = {}
start = time.perf_counter()
markdown_text = chr(10).join(["### Question", QUESTION, "", "_Streaming agent answer..._"])
answer_display = display(Markdown(markdown_text), display_id=True)

for event in rna_seq_agent.stream(
    {"messages": [{"role": "user", "content": QUESTION}]},
    stream_mode="values",
):
    final_message = event["messages"][-1]
    tool_calls = getattr(final_message, "tool_calls", None)
    if tool_calls:
        print("tool_calls:", tool_calls)

    content = final_message.content
    answer_text = str(content)
    markdown_text = chr(10).join(["### Question", QUESTION, "", "### Agent answer", answer_text])
    answer_display.update(Markdown(markdown_text))

    if getattr(final_message, "usage_metadata", None):
        usage = final_message.usage_metadata

print("seconds:", round(time.perf_counter() - start, 3))
print("total_tokens:", usage.get("total_tokens") if usage else None)